In [1]:
import json
import os
from typing import List, Tuple
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from geopy.distance import geodesic
from torch.distributions import Categorical
from copy import deepcopy

In [2]:
if torch.cuda.is_available():
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU not available. Please set Runtime > Change runtime type > GPU.")


❌ GPU not available. Please set Runtime > Change runtime type > GPU.


In [3]:
# data_loader.py
# ---------------------------
# Loader untuk user_coords, evacuation candidates, dan MMI dari file JSON

# ===== Load User Coordinates dari JSON =====
def load_user_coords(path: str) -> List[Tuple[float, float]]:
    with open(path, "r") as f:
        data = json.load(f)
    coords = [(entry["latitude"], entry["longitude"]) for entry in data]
    return coords


# ===== Load Dataset Event (Evacuation + MMI) =====
def load_event_dataset(event_id: str, dataset_dir: str):
    path = os.path.join(dataset_dir, f"2025-04-03-dataset-{event_id}.json")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset untuk event {event_id} tidak ditemukan.")

    with open(path, "r") as f:
        data = json.load(f)

    # Ambil titik evakuasi sebagai kandidat
    evac_points = [
        (d["latitude"], d["longitude"]) for d in data.get("evacuationData", [])
    ]

    # Format ulang MMI untuk fungsi RL
    mmi_points = []
    for m in data.get("mmiData", []):
        mmi_points.append(
            {"lat": m["latitude"], "lon": m["longitude"], "mmi": m["mmi_level"]}
        )

    return evac_points, mmi_points


In [4]:
# meta_rl_reinforce_baseline.py
# -----------------------------------------
# RL sederhana (REINFORCE) per event (task)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GAMMA = 0.99
LEARNING_RATE = 1e-3
MAX_EPISODES = 100
INNER_LR = 1e-2
INNER_STEPS = 5

# ===== Policy Network =====
class PolicyNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, output_dim=100):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)

def prepare_mmi_tensors(mmi_points):
    coords = torch.tensor([[p["lat"], p["lon"]] for p in mmi_points], dtype=torch.float32).to(DEVICE)
    values = torch.tensor([p["mmi"] for p in mmi_points], dtype=torch.float32).to(DEVICE)
    return coords, values

def get_nearest_mmi_tensor(lat, lon, mmi_coords, mmi_values):
    user = torch.tensor([lat, lon], dtype=torch.float32).to(DEVICE)
    dists = torch.norm(mmi_coords - user, dim=1)
    idx = torch.argmin(dists)
    return mmi_values[idx], dists[idx] * 111.0

def compute_reward(user_lat, user_lon, evac_lat, evac_lon, mmi_coords, mmi_values):
    mmi, dist = get_nearest_mmi_tensor(evac_lat, evac_lon, mmi_coords, mmi_values)
    if mmi < 4:
        reward = 10.0 - dist
    else:
        reward = -10.0
    return reward.item(), mmi.item()

def rollout(model, user_coords, evac_candidates, mmi_coords, mmi_values):
    log_probs, rewards = [], []
    for user_lat, user_lon in user_coords:
        state = torch.tensor([user_lat, user_lon], dtype=torch.float32).to(DEVICE)
        logits = model(state)
        probs = torch.softmax(logits, dim=0)
        dist = Categorical(probs)
        action = dist.sample()
        evac_lat, evac_lon = evac_candidates[action.item()]
        reward, _ = compute_reward(user_lat, user_lon, evac_lat, evac_lon, mmi_coords, mmi_values)
        log_probs.append(dist.log_prob(action))
        rewards.append(reward)
    return log_probs, rewards

def compute_loss(log_probs, rewards):
    returns, G = [], 0
    for r in reversed(rewards):
        G = r + GAMMA * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32).to(DEVICE)
    log_probs = torch.stack(log_probs)
    loss = -(log_probs * returns).sum()
    return loss

def adapt(model, user_coords, evac_candidates, mmi_coords, mmi_values):
    model_adapted = deepcopy(model)
    optimizer = optim.SGD(model_adapted.parameters(), lr=INNER_LR)
    for _ in range(INNER_STEPS):
        log_probs, rewards = rollout(model_adapted, user_coords, evac_candidates, mmi_coords, mmi_values)
        loss = compute_loss(log_probs, rewards)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model_adapted

def meta_train(tasks):
    meta_model = PolicyNetwork(input_dim=2, output_dim=len(tasks[0]["evac_candidates"])).to(DEVICE)
    meta_optimizer = optim.Adam(meta_model.parameters(), lr=META_LR)

    for iteration in range(MAX_META_ITER):
        meta_grads = [torch.zeros_like(p) for p in meta_model.parameters()]
        for task in tasks:
            model_adapted = adapt(meta_model, task["user_coords"], task["evac_candidates"], task["mmi_coords"], task["mmi_values"])
            log_probs, rewards = rollout(model_adapted, task["user_coords"], task["evac_candidates"], task["mmi_coords"], task["mmi_values"])
            loss = compute_loss(log_probs, rewards)
            grads = torch.autograd.grad(loss, model_adapted.parameters())
            for i, g in enumerate(grads):
                meta_grads[i] += g.detach()
        for p, g in zip(meta_model.parameters(), meta_grads):
            p.grad = g / len(tasks)
        meta_optimizer.step()
        meta_optimizer.zero_grad()
        print(f"📈 Iterasi Meta-RL {iteration+1}/{MAX_META_ITER} selesai")

    return meta_model

def evaluate_model(model, user_coords, evac_candidates, mmi_coords, mmi_values):
    correct = 0
    results = []
    with torch.no_grad():
        for user_lat, user_lon in user_coords:
            state = torch.tensor([user_lat, user_lon], dtype=torch.float32).to(DEVICE)
            logits = model(state)
            probs = torch.softmax(logits, dim=0).cpu().numpy()
            top_choice = int(np.argmax(probs))
            evac_lat, evac_lon = evac_candidates[top_choice]
            reward, mmi = compute_reward(user_lat, user_lon, evac_lat, evac_lon, mmi_coords, mmi_values)
            results.append({
                "user_coord": [user_lat, user_lon],
                "evac_coord": [evac_lat, evac_lon],
                "mmi": mmi,
                "reward": reward,
                "valid_mmi": int(mmi < 4)
            })
            correct += int(mmi < 4)
    accuracy = correct / len(user_coords)
    return results, accuracy

# Gabungkan ke log global multi-event
def save_to_multi_event_log(event_id, logs, output_path="rl_logs_multi_event.json"):
    if os.path.exists(output_path):
        with open(output_path, "r") as f:
            all_logs = json.load(f)
    else:
        all_logs = {}

    all_logs[event_id] = logs

    with open(output_path, "w") as f:
        json.dump(all_logs, f, indent=2)
    print(f"📁 Log event '{event_id}' disimpan ke '{output_path}'")


# ===== Training per Event =====
def adapt_and_log(meta_model, user_coords, evac_candidates, mmi_points, event_id, output_path="rl_logs_meta_event.json"):
    model = deepcopy(meta_model)
    optimizer = optim.SGD(model.parameters(), lr=INNER_LR)
    logs = {"episodes": [], "final_evaluation": []}
    mmi_coords, mmi_values = prepare_mmi_tensors(mmi_points)

    for episode in range(INNER_STEPS):
        log_probs, rewards = [], []
        episode_log = {"steps": [], "total_reward": 0, "loss": 0}

        for user_lat, user_lon in user_coords:
            state = torch.tensor([user_lat, user_lon], dtype=torch.float32).to(DEVICE)
            logits = model(state)
            probs = torch.softmax(logits, dim=0)
            dist = Categorical(probs)
            action = dist.sample()

            evac_lat, evac_lon = evac_candidates[action.item()]
            reward, mmi = compute_reward(user_lat, user_lon, evac_lat, evac_lon, mmi_coords, mmi_values)

            log_probs.append(dist.log_prob(action))
            rewards.append(reward)

            step_log = {
                "user_coord": [user_lat, user_lon],
                "evac_coord": [evac_lat, evac_lon],
                "mmi": float(mmi),
                "reward": float(reward),
                "action": int(action.item()),
            }
            episode_log["steps"].append(step_log)

        returns, G = [], 0
        for r in reversed(rewards):
            G = r + GAMMA * G
            returns.insert(0, G)
        returns = torch.tensor(returns, dtype=torch.float32).to(DEVICE)
        loss = -(torch.stack(log_probs) * returns).sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        episode_log["total_reward"] = float(sum(rewards))
        episode_log["loss"] = float(loss.item())
        logs["episodes"].append(episode_log)

    # ==== Evaluasi akhir ====
    eval_results, accuracy = evaluate_model(model, user_coords, evac_candidates, mmi_coords, mmi_values)
    logs["final_evaluation"] = eval_results
    logs["accuracy"] = accuracy

    save_to_multi_event_log(event_id, logs, output_path)
    print(f"📁 Evaluasi dan log disimpan untuk event '{event_id}'")

    return model


def evaluate_all(model, task_list, output_log="rl_logs_meta_eval.json"):
    all_logs = {}
    for task in task_list:
        user_coords = task["user_coords"]
        evac_candidates = task["evac_candidates"]
        mmi_coords = task["mmi_coords"]
        mmi_values = task["mmi_values"]

        results, accuracy = evaluate_model(
            model, user_coords, evac_candidates, mmi_coords, mmi_values
        )
        all_logs[task["event_id"]] = {"final_evaluation": results, "accuracy": accuracy}

    with open(output_log, "w") as f:
        json.dump(all_logs, f, indent=2)
    print(f"✅ Semua hasil evaluasi disimpan di '{output_log}'")

In [5]:
# ===== Konfigurasi =====
DATA_DIR = "/kaggle/input/dataset-training-meta-rl-ada-gempa"
USER_COORDS_PATH = os.path.join(DATA_DIR, "user_locations.json")
EVENT_LIST = [
    # "usp000gjww",
    # "us6000m2rj",
    # "us7000beax",
    # "us7000hgtm",
    # "us7000lu24",
    "us10002m8b",
    "us1000741g",
    "usc000mspp",
    "usp000esnf",
    "usp000f6vz",
]
SAVE_MODEL_DIR = "/kaggle/working"
os.makedirs(SAVE_MODEL_DIR, exist_ok=True)

# ===== Load data user global =====
user_coords = load_user_coords(USER_COORDS_PATH)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/dataset-training-meta-rl-ada-gempa\\user_locations.json'

In [ ]:
# ===== Fungsi bantu: memuat data dan membuat task list =====
def build_task_list(event_list, data_dir, user_coords_path):
    with open(user_coords_path, "r") as f:
        user_coords = json.load(f)

    tasks = []

    for event_id in event_list:
        try:
            evac_path = os.path.join(data_dir, f"evacuation_points_{event_id}.json")
            mmi_path = os.path.join(data_dir, f"cont_mmi_{event_id}.json")

            with open(evac_path, "r") as f:
                evac_data = json.load(f)
            evac_candidates = [[p["lat"], p["lon"]] for p in evac_data]

            with open(mmi_path, "r") as f:
                mmi_data = json.load(f)
            mmi_coords = torch.tensor([[p["lat"], p["lon"]] for p in mmi_data], dtype=torch.float32).to(DEVICE)
            mmi_values = torch.tensor([p["mmi"] for p in mmi_data], dtype=torch.float32).to(DEVICE)

            task = {
                "event_id": event_id,
                "user_coords": user_coords,
                "evac_candidates": evac_candidates,
                "mmi_coords": mmi_coords,
                "mmi_values": mmi_values,
            }
            tasks.append(task)
        except Exception as e:
            print(f"[SKIP] Gagal memuat data untuk {event_id}: {e}")

    return tasks


In [ ]:
def meta_train_loop(task_list, model_save_dir="models_meta", max_iter=50):
    os.makedirs(model_save_dir, exist_ok=True)
    meta_model = PolicyNetwork(input_dim=2, output_dim=len(task_list[0]["evac_candidates"])).to(DEVICE)
    meta_optimizer = optim.Adam(meta_model.parameters(), lr=LEARNING_RATE)

    for iteration in range(max_iter):
        meta_grads = [torch.zeros_like(p) for p in meta_model.parameters()]
        print(f"\n[Meta-Iter {iteration+1}/{max_iter}]")

        for task in task_list:
            try:
                adapted_model = adapt(meta_model,
                                      task["user_coords"],
                                      task["evac_candidates"],
                                      task["mmi_coords"],
                                      task["mmi_values"])
                log_probs, rewards = rollout(adapted_model,
                                             task["user_coords"],
                                             task["evac_candidates"],
                                             task["mmi_coords"],
                                             task["mmi_values"])
                loss = compute_loss(log_probs, rewards)
                grads = torch.autograd.grad(loss, adapted_model.parameters())

                for i, g in enumerate(grads):
                    meta_grads[i] += g.detach()

            except Exception as e:
                print(f"[SKIP] {task['event_id']}: {e}")

        for p, g in zip(meta_model.parameters(), meta_grads):
            p.grad = g / len(task_list)

        meta_optimizer.step()
        meta_optimizer.zero_grad()

        print("✅ Meta-update selesai")

    final_model_path = os.path.join(model_save_dir, "meta_model_final.pt")
    torch.save(meta_model.state_dict(), final_model_path)
    print(f"\n✅ Meta-model disimpan di: {final_model_path}")

    return meta_model


In [ ]:
TASK_LIST = build_task_list(EVENT_LIST, DATA_DIR, USER_COORDS_PATH)
meta_model = meta_train_loop(TASK_LIST, model_save_dir=SAVE_MODEL_DIR, max_iter=50)
evaluate_all(meta_model, TASK_LIST, output_log="rl_logs_meta_eval.json")

In [ ]:
print("\n[BRAIN] Menggabungkan bobot semua model menjadi Meta-Model...")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

evac_size = all_models[0]["fc2.weight"].shape[0]
meta_model = PolicyNetwork(input_dim=2, output_dim=evac_size).to(DEVICE)
meta_state = meta_model.state_dict()

for key in meta_state:
    meta_state[key] = torch.stack(
        [model[key].to(DEVICE) for model in all_models]
    ).mean(0)

meta_model.load_state_dict(meta_state)
torch.save(meta_model.state_dict(), os.path.join(SAVE_MODEL_DIR, "meta_model_final-full-30-05-2025(tunning-2).pt"))
print("\n✅ Meta-model akhir disimpan sebagai 'meta_model_final-full-30-05-2025(tunning-2).pt'")


In [ ]:
import json
import matplotlib.pyplot as plt

# Ganti path sesuai lokasi file JSON kamu
json_path = "rl_logs_multi_event.json"

with open(json_path, "r") as f:
    all_logs = json.load(f)

event_ids = []
avg_rewards = []
accuracies = []

for event_id, log in all_logs.items():
    total_rewards = [ep["total_reward"] for ep in log["episodes"]]
    avg_reward = sum(total_rewards) / len(total_rewards)
    acc = log.get("accuracy", 0)
    
    event_ids.append(event_id)
    avg_rewards.append(avg_reward)
    accuracies.append(acc)

# 📈 Plot Rata-rata Total Reward
plt.figure(figsize=(12, 5))
plt.bar(event_ids, avg_rewards, color='dodgerblue')
plt.title("Rata-rata Total Reward per Event")
plt.xlabel("Event ID")
plt.ylabel("Average Total Reward")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.tight_layout()
plt.show()

# 🎯 Plot Akurasi (MMI < 4)
plt.figure(figsize=(12, 5))
plt.bar(event_ids, accuracies, color='seagreen')
plt.title("Akurasi RL per Event (MMI < 4)")
plt.xlabel("Event ID")
plt.ylabel("Akurasi")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# CONVERT TO ONNX

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PolicyNetwork(input_dim=2, output_dim=270).to(DEVICE)
model.load_state_dict(torch.load("/kaggle/working/meta_model_final-full-30-05-2025(tunning-2).pt", map_location=DEVICE))
model.eval()


In [ ]:
import torch.onnx

# Dummy input: user location [lat, lon]
dummy_input = torch.randn(1, 2).to(DEVICE)  # 1 sample, 2 fitur (lat, lon)

torch.onnx.export(
    model,                      # model PyTorch
    dummy_input,                # contoh input
    "meta_model_final-full-30-05-2025(tunning-2).onnx",   # nama file output
    export_params=True,
    opset_version=11,          # versi aman untuk ONNX Runtime
    do_constant_folding=True,  # optimasi
    input_names=["user_location"],
    output_names=["evacuation_logits"],
    dynamic_axes={
        "user_location": {0: "batch_size"},
        "evacuation_logits": {0: "batch_size"}
    }
)

print("✅ Model berhasil dikonversi ke ONNX → meta_model_final.onnx")
